# CertVIC -- InternVL2-8B eval on Kaggle (SELF-DOWNLOAD, no weights input)

Downloads `OpenGVLab/InternVL2-8B` from Hugging Face at runtime (Internet ON), pins a
Transformers stack that works with InternVL2's remote code, loads it in **bf16 sharded across
2x T4** (no bitsandbytes), then runs CertVIC eval over the pilot pairs via the in-process
`certvic.eval.run_eval` (leakage / evidence / resume / manifest gates intact,
`provider_name=internvl_8b`).

**You attach only 3 inputs:** CertVIC code, presence data, absent-object control data.
**No InternVL weights dataset.** Outputs: `pred_internvl_8b_{presence,control}_merged.jsonl`
(+ `internvl_preds.zip`). Pilot-only; no paper claims.

> Fixes two Kaggle breakages: (1) `AttributeError: ... 'all_tied_weights_keys'` (Transformers
> >=4.50 vs InternVL2 remote code) -> we pin `transformers==4.37.2` **before** importing it;
> (2) `No module named 'triton.ops'` / missing `libbitsandbytes_cuda128.so` (bitsandbytes is
> broken on the Py3.12 / CUDA 12.8 image) -> we **avoid bitsandbytes** and load bf16 across both
> T4s. Single-GPU falls back to 4-bit (best-effort) -- prefer **T4 x2**.

## Settings & inputs

- **Accelerator: GPU T4 x2 (recommended).** InternVL2-8B in bf16 is ~16 GB -- it does NOT fit on
  one 16 GB T4, but fits sharded across two. A single T4/P100 falls back to 4-bit (needs a working
  bitsandbytes; less reliable on new images). **Internet: ON** (for the model download).
- **Attach 3 datasets** (any mount names -- auto-detected):
  1. **CertVIC code** -- contains the `certvic/` package dir (e.g. `dist/certvic_kaggle_main200_bundle.zip` or the repo).
  2. **presence data** -- `dist/certvic_main200_session2_data.zip` (its `pilot_eval_tasks_reviewed.jsonl` has **91** tasks).
  3. **absent-object control** -- `dist/certvic_absent_object_control.zip` (its `pilot_eval_tasks_reviewed.jsonl` has **120** tasks).
- Optional: add an `HF_TOKEN` Kaggle Secret (not required -- InternVL2-8B is public).
- Run top-to-bottom. **Stop at the smoke-test cell** and confirm yes/no before the full run.

In [ ]:
# CELL 1 -- pin the InternVL2-compatible stack BEFORE importing transformers.
# (transformers>=4.50 breaks InternVL2 remote code: 'all_tied_weights_keys' / GenerationMixin.)
# bitsandbytes is deliberately NOT pinned here: the default bf16 multi-GPU path needs none, and
# 0.43.x is broken on the Py3.12/CUDA-12.8 image (triton.ops removed). The 1-GPU 4-bit fallback
# installs a current bitsandbytes on demand.
import sys, subprocess, os
PINS = [
    "transformers==4.37.2",
    "tokenizers==0.15.2",
    "huggingface_hub==0.23.4",
    "accelerate==0.30.1",
    "timm==0.9.12",
    "einops",
    "sentencepiece",
]
_SENTINEL = "/kaggle/working/.deps_installed_internvl"
if not os.path.exists(_SENTINEL):
    print("installing pinned stack (one-time)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    open(_SENTINEL, "w").write("ok")
    print("done.")
else:
    print("deps already installed (sentinel present).")

if "transformers" in sys.modules and not getattr(sys.modules["transformers"], "__version__", "").startswith("4.37"):
    raise SystemExit("Stale transformers already imported -> Run menu > 'Restart & Run All' once. "
                     "(The sentinel makes the reinstall instant on the second pass.)")

In [ ]:
# CELL 2 -- verify the pinned versions are the ones now active.
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| CUDA", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")
assert transformers.__version__.startswith("4.37"), (
    f"transformers {transformers.__version__} active, need 4.37.2 -> Run > 'Restart & Run All'.")
print("version stack OK")

In [ ]:
# CELL 3 -- auto-detect Kaggle inputs (override only if auto-detect fails).
import glob, os, json

CERTVIC_DIR    = None   # optional override: parent dir that contains 'certvic/'
PRESENCE_INPUT = None   # optional override: dir with the 91-task pilot_eval_tasks_reviewed.jsonl
CONTROL_INPUT  = None   # optional override: dir with the 120-task pilot_eval_tasks_reviewed.jsonl

def _find_certvic_parent():
    for p in glob.glob("/kaggle/input/**/certvic", recursive=True):
        if os.path.isdir(p) and os.path.exists(os.path.join(p, "eval", "run_eval.py")):
            return os.path.dirname(p)
    return None

def _find_task_bundles():
    pres = ctrl = None
    for p in glob.glob("/kaggle/input/**/pilot_eval_tasks_reviewed.jsonl", recursive=True):
        try:
            n = sum(1 for _ in open(p))
        except OSError:
            continue
        if n == 91 and pres is None:
            pres = os.path.dirname(p)
        elif n == 120 and ctrl is None:
            ctrl = os.path.dirname(p)
    return pres, ctrl

CERTVIC_DIR = CERTVIC_DIR or _find_certvic_parent()
_p, _c = _find_task_bundles()
PRESENCE_INPUT = PRESENCE_INPUT or _p
CONTROL_INPUT  = CONTROL_INPUT or _c
MODEL_DIR = "/kaggle/working/hf_models/internvl2-8b"

print("CERTVIC_DIR   :", CERTVIC_DIR)
print("PRESENCE_INPUT:", PRESENCE_INPUT, "(91-task bundle)")
print("CONTROL_INPUT :", CONTROL_INPUT, "(120-task bundle)")
print("MODEL_DIR     :", MODEL_DIR, "(downloaded at runtime)")

_missing = [n for n, v in [("CERTVIC code", CERTVIC_DIR), ("presence data", PRESENCE_INPUT),
                           ("absent-object control data", CONTROL_INPUT)] if not v]
if _missing:
    raise RuntimeError("Missing required input(s): " + ", ".join(_missing) +
                       ". Attach the 3 datasets (see the prereqs cell) or set the override variables.")
sys.path.insert(0, CERTVIC_DIR)
import certvic; print("certvic OK from", os.path.dirname(certvic.__file__))

In [ ]:
# CELL 4 -- download InternVL2-8B at runtime (cached + resumable). Internet must be ON.
import os
from huggingface_hub import snapshot_download

def _hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")  # optional; InternVL2-8B is public

os.makedirs(MODEL_DIR, exist_ok=True)
if os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("reusing cached weights at", MODEL_DIR)
else:
    print("downloading OpenGVLab/InternVL2-8B -> ", MODEL_DIR, "(~16 GB, one time)...")
    snapshot_download(repo_id="OpenGVLab/InternVL2-8B", local_dir=MODEL_DIR, token=_hf_token())
    print("download complete.")
assert os.path.exists(os.path.join(MODEL_DIR, "config.json")), "snapshot missing config.json"

In [ ]:
# CELL 5 -- CertVIC run config (label only; weights are loaded from MODEL_DIR in the patch cell).
WORK = "/kaggle/working"
CFG = f"{WORK}/kaggle_internvl.yaml"
cfg_lines = [
    "mode: kaggle_open_vlm", "provider: internvl_8b", "model_id: OpenGVLab/InternVL2-8B",
    "device: cuda", "dtype: bfloat16", "batch_size: 1",
    "max_new_tokens: 16", "temperature: 0.0", "paid_services_enabled: false",
]
open(CFG, "w").write("\n".join(cfg_lines) + "\n")
print("wrote", CFG)

In [ ]:
# CELL 6 -- official InternVL2 preprocessing + load (bf16 sharded on >=2 GPUs; 4-bit on 1 GPU) + patch.
import math, sys, subprocess
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from PIL import Image
from transformers import AutoModel, AutoTokenizer
import certvic.providers.open_vlm as ovlm

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
MAX_TILES = 1   # 1 tile (448px) is fast and enough for yes/no presence; raise to 6/12 for fine detail

def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff, best_ratio = float("inf"), (1, 1)
    area = width * height
    for ratio in target_ratios:
        target = ratio[0] / ratio[1]
        diff = abs(aspect_ratio - target)
        if diff < best_ratio_diff:
            best_ratio_diff, best_ratio = diff, ratio
        elif diff == best_ratio_diff and area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
            best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    w, h = image.size
    aspect_ratio = w / h
    target_ratios = sorted(
        {(i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1)
         if min_num <= i * j <= max_num}, key=lambda x: x[0] * x[1])
    tar = find_closest_aspect_ratio(aspect_ratio, target_ratios, w, h, image_size)
    tw, th = image_size * tar[0], image_size * tar[1]
    blocks = tar[0] * tar[1]
    resized = image.resize((tw, th))
    cols = tw // image_size
    tiles = []
    for i in range(blocks):
        box = ((i % cols) * image_size, (i // cols) * image_size,
               ((i % cols) + 1) * image_size, ((i // cols) + 1) * image_size)
        tiles.append(resized.crop(box))
    if use_thumbnail and len(tiles) != 1:
        tiles.append(image.resize((image_size, image_size)))
    return tiles

def load_image(image_file, input_size=448, max_num=MAX_TILES):
    image = Image.open(image_file).convert("RGB")
    transform = build_transform(input_size)
    tiles = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    return torch.stack([transform(t) for t in tiles])

def split_model(num_layers=32):
    # Official InternVL multi-GPU device_map: GPU0 also holds the ViT, so it takes fewer LLM layers.
    n = torch.cuda.device_count()
    if n <= 1:
        return None
    per = math.ceil(num_layers / (n - 0.5))
    alloc = [per] * n
    alloc[0] = math.ceil(per * 0.5)
    dm, cnt = {}, 0
    for i, a in enumerate(alloc):
        for _ in range(a):
            if cnt < num_layers:
                dm[f"language_model.model.layers.{cnt}"] = i
                cnt += 1
    for k in ["vision_model", "mlp1", "language_model.model.tok_embeddings",
              "language_model.model.embed_tokens", "language_model.output",
              "language_model.model.norm", "language_model.model.rotary_emb",
              "language_model.lm_head", f"language_model.model.layers.{num_layers - 1}"]:
        dm[k] = 0
    return dm

# flash-attn is optional; InternVL falls back to eager attention if it is missing (the warning is harmless).
_ngpu = torch.cuda.device_count()
_common = dict(torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True)
if _ngpu >= 2:
    print(f"{_ngpu} GPUs -> bf16 sharded via split_model (no bitsandbytes).")
    model = AutoModel.from_pretrained(MODEL_DIR, device_map=split_model(), **_common).eval()
else:
    print("Only 1 GPU -> InternVL2-8B bf16 (~16 GB) will not fit a 16 GB T4; trying 4-bit. "
          "Prefer 'GPU T4 x2' for the robust bf16 path.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes>=0.45.0"], check=True)
    from transformers import BitsAndBytesConfig
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModel.from_pretrained(MODEL_DIR, quantization_config=qcfg, device_map={"": 0}, **_common).eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True, use_fast=False)
GEN = dict(max_new_tokens=16, do_sample=False)   # deterministic

import time as _time
_PROG = {"n": 0, "tag": "", "t0": None}
@torch.inference_mode()
def _internvl_answer(self, image_path, prompt):
    if _PROG["t0"] is None:
        _PROG["t0"] = _time.time()
    pv = load_image(image_path).to(torch.bfloat16).cuda()
    out = str(model.chat(tokenizer, pv, "<image>\n" + prompt, GEN)).strip()
    _PROG["n"] += 1
    if _PROG["n"] <= 2 or _PROG["n"] % 20 == 0:
        rate = _PROG["n"] / max(_time.time() - _PROG["t0"], 1e-6)
        print(f"  {_PROG['tag']}: {_PROG['n']} generations | {rate:.2f} gen/s", flush=True)
    return out

ovlm.OpenVLMProvider.load = lambda self: None
ovlm.OpenVLMProvider.answer = _internvl_answer
print("InternVL2-8B loaded + OpenVLMProvider.answer patched; MAX_TILES =", MAX_TILES, "| GPUs:", _ngpu)

In [ ]:
# CELL 7 -- SMOKE TEST: 2 presence + 2 control examples must parse to yes/no.
import json
from certvic.eval.parse import parse_answer

def smoke(job_input, label, k=2):
    # Resolve paths like the run cell: {input}/<f> = edited, {input}/orig/<f> = original.
    rows = [json.loads(l) for l in open(f"{job_input}/pilot_eval_tasks_reviewed.jsonl")][:k]
    ok = True
    for r in rows:
        imgs = {"original": f"{job_input}/orig/{os.path.basename(r['original_image_path'])}",
                "edited":   f"{job_input}/{os.path.basename(r['edited_image_path'])}"}
        for variant, img in imgs.items():
            raw = _internvl_answer(None, img, r["question_original"])
            p = parse_answer(raw, "yes_no", strict=True)
            print(f"[{label}] {r['item_id'][:20]:20s} {variant:8s} raw={raw!r:12s} "
                  f"parsed={p.parsed_answer} ok={p.parse_ok}")
            ok = ok and p.parse_ok and p.parsed_answer in ("yes", "no")
    return ok

_ok = smoke(PRESENCE_INPUT, "presence") and smoke(CONTROL_INPUT, "control")
if not _ok:
    raise RuntimeError("Smoke test produced non-yes/no answers -- fix the model/chat patch before the full run.")
print("SMOKE OK -- answers parse to yes/no. Proceed.")

In [ ]:
# CELL 8 -- remap bundle image paths, project to the strict TaskItem schema, then run CertVIC eval.
import json, time
from certvic.eval.run_eval import run_eval
from certvic.schema import TaskItem
from certvic.schema.edit import EditSpec
from certvic.schema.source import SourceImageRecord

def _to_taskitem(r):
    # Project the flat reviewed/preview schema onto the nested TaskItem run_eval requires.
    src = SourceImageRecord(source_id=r['source_id'], source_name='ADE20K', license_category='pointer_only')
    ed = EditSpec(edit_id=r['edit_id'], source_id=r['source_id'], edit_type=r['edit_type'],
                  task_family=r['task_family'], domain=r['domain'], expected_effect=r['expected_effect'])
    m = dict(r.get('metadata') or {})
    m.setdefault('evidence_status', r.get('evidence_status', 'HUMAN_REVIEWED_NON_EVIDENCE'))
    return TaskItem(item_id=r['item_id'], source=src, edit=ed,
                    original_image_path=r['original_image_path'], edited_image_path=r['edited_image_path'],
                    question_original=r['question_original'], question_edited=r['question_edited'],
                    answer_original=r['answer_original'], answer_edited=r['answer_edited'],
                    required_change=r['required_change'], answer_format=r['answer_format'],
                    task_family=r['task_family'], domain=r['domain'], split=r['split'], metadata=m)

def remap_tasks(job_input, name):
    rows = [json.loads(l) for l in open(f'{job_input}/pilot_eval_tasks_reviewed.jsonl')]
    miss = 0
    for r in rows:
        ob, eb = os.path.basename(r['original_image_path']), os.path.basename(r['edited_image_path'])
        r['original_image_path'] = f'{job_input}/orig/{ob}'
        r['edited_image_path']   = f'{job_input}/{eb}'
        miss += (not os.path.exists(r['original_image_path'])) + (not os.path.exists(r['edited_image_path']))
    assert miss == 0, f'{name}: {miss} images not found -- check bundle layout/path.'
    # presence bundle ships the FLAT reviewed schema -> project; control bundle is already nested -> pass through.
    needs_convert = bool(rows) and 'source' not in rows[0]
    if needs_convert:
        rows = [json.loads(_to_taskitem(r).model_dump_json()) for r in rows]
    dst = f'{WORK}/tasks_{name}.jsonl'
    open(dst, 'w').writelines(json.dumps(r) + '\n' for r in rows)
    print(f'{name}: {len(rows)} tasks -> {dst} | schema: ' + ('flat->TaskItem' if needs_convert else 'already nested'))
    return dst

JOBS = [
    {'name': 'presence', 'input': PRESENCE_INPUT, 'run_id': 'main200_internvl_8b_presence',
     'out': 'pred_internvl_8b_presence_merged.jsonl'},
    {'name': 'control', 'input': CONTROL_INPUT, 'run_id': 'main200_internvl_8b_control',
     'out': 'pred_internvl_8b_control_merged.jsonl'},
]
for job in JOBS:
    tasks = remap_tasks(job['input'], job['name'])
    out = f"{WORK}/{job['out']}"
    _PROG['n'], _PROG['tag'], _PROG['t0'] = 0, job['name'], None
    print(f"running {job['name']} ...", flush=True)
    t0 = time.time()
    summary = run_eval(config_path=CFG, tasks_path=tasks, out_path=out,
                       provider_name='internvl_8b', run_id=job['run_id'],
                       num_shards=1, strict_leakage=True, evidence_run=True,
                       fail_fast=False, overwrite=False)
    print(job['name'], summary, f"({time.time()-t0:.0f}s) -> {out}")

In [ ]:
# CELL 9 -- final parse rate + yes/no distribution + provider stamp.
import collections, json
for job in JOBS:
    rows = [json.loads(l) for l in open(f"{WORK}/{job['out']}")]
    ans = collections.Counter(r["parsed_answer"] for r in rows)
    okr = sum(r["parse_ok"] for r in rows) / len(rows)
    print(f"{job['name']:9s}: {len(rows):3d} preds | parse_ok={okr:.3f} | answers={dict(ans)} "
          f"| provider={sorted({r['provider_name'] for r in rows})}")

In [ ]:
# CELL 10 -- zip predictions + manifests for download.
import glob, zipfile
files = sorted(glob.glob(f"{WORK}/pred_internvl_8b_*_merged.jsonl")
               + glob.glob(f"{WORK}/pred_internvl_8b_*_merged.jsonl.run_manifest.json"))
with zipfile.ZipFile(f"{WORK}/internvl_preds.zip", "w") as z:
    for f in files:
        z.write(f, os.path.basename(f))
print("wrote internvl_preds.zip ->", [os.path.basename(f) for f in files])

## Back on the Mac

Download `pred_internvl_8b_presence_merged.jsonl` and `pred_internvl_8b_control_merged.jsonl`
(or `internvl_preds.zip`), then:

```bash
cd /path/to/certVIC
python3 scripts/pilot_report_from_raw.py \
  --provider internvl_8b --model-name OpenGVLab/InternVL2-8B --run-label internvl_8b \
  --raw-presence /path/to/pred_internvl_8b_presence_merged.jsonl \
  --raw-control  /path/to/pred_internvl_8b_control_merged.jsonl
```

Writes `data/results/main_real_200/pilot_report__internvl_8b/` (+ sha256-locked raw) and
refreshes `multimodel_pilot_summary.{md,csv,json}`. REFUSES if a file is missing or its
`provider_name` != internvl_8b. Pilot-only; no paper-grade claim.